In [2]:
import pulp

# Actividad en clase

## SunRay

In [6]:
I = [1, 2, 3] # Silo (Filas)
J = [1, 2, 3, 4] # Molino (Columnas)

c = [
    [10, 2, 20, 11],
    [12, 7, 9, 20],
    [4, 14, 16, 18]
]

d = [5, 15, 15, 15] # Demanda de los 4 Molinos
o = [15, 25, 10]    # Oferta de los 3 Silos

model = pulp.LpProblem("SunRay_Transport", pulp.LpMinimize)

# Variable de decisión: Cantidad enviada del Silo i al Molino j
x = {(i, j): pulp.LpVariable(f"Envio_Silo_{i}_a_Molino_{j}", lowBound=0, cat='Continuous') for i in I for j in J}

# 2. Función Objetivo: Nota el i-1 y j-1 para respetar el índice 0 de Python
model += pulp.lpSum(x[i, j] * c[i-1][j-1] for i in I for j in J), "Costo_Total"

In [7]:
for i in I:
    model += pulp.lpSum(x[i, j] for j in J) <= o[i-1], f"Oferta_Silo_{i}"

# 4. Restricción de Demanda: Lo que llega al Molino 'j' debe ser igual a su demanda
for j in J:
    model += pulp.lpSum(x[i, j] for i in I) == d[j-1], f"Demanda_Molino_{j}"

# Resolver el problema
model.solve()

# Imprimir los resultados
print(f"Estado del modelo: {pulp.LpStatus[model.status]}\n")
print("--- Plan de Envíos Óptimo ---")
for i in I:
    for j in J:
        if x[i, j].varValue > 0:
            print(f"Enviar {x[i, j].varValue} camiones del Silo {i} al Molino {j}")

print(f"\nCosto Total Mínimo: ${pulp.value(model.objective) * 100} dólares")

Estado del modelo: Optimal

--- Plan de Envíos Óptimo ---
Enviar 5.0 camiones del Silo 1 al Molino 2
Enviar 10.0 camiones del Silo 1 al Molino 4
Enviar 10.0 camiones del Silo 2 al Molino 2
Enviar 15.0 camiones del Silo 2 al Molino 3
Enviar 5.0 camiones del Silo 3 al Molino 1
Enviar 5.0 camiones del Silo 3 al Molino 4

Costo Total Mínimo: $43500.0 dólares


# Tarea

In [25]:
I = [1, 2, 3] # Refinerias
J = [1, 2, 3] # Áreas de distribución

c = [
    [120, 180, 9999],
    [300, 100, 80],
    [200, 250, 120]
]

d = [4, 8, 7]
o = [6, 5, 8]
model = pulp.LpProblem("Refineria", pulp.LpMinimize)

# Variable de decisión: Cantidad enviada del Silo i al Molino j
x = {(i, j): pulp.LpVariable(f"Envio_Refineria_{i}_a_AD_{j}", lowBound=0, cat='Continuous') for i in I for j in J}

# 2. Función Objetivo: Nota el i-1 y j-1 para respetar el índice 0 de Python
model += pulp.lpSum(x[i, j] * c[i-1][j-1] for i in I for j in J), "Costo_Total"

In [26]:
for i in I:
    model += pulp.lpSum(x[i, j] for j in J) <= o[i-1], f"Oferta_Refineria_{i}"

# 4. Restricción de Demanda: Lo que llega al Molino 'j' debe ser igual a su demanda
for j in J:
    model += pulp.lpSum(x[i, j] for i in I) == d[j-1], f"Demanda_AD_{j}"

# Resolver el problema
model.solve()

# Imprimir los resultados
print(f"Estado del modelo: {pulp.LpStatus[model.status]}\n")
print("--- Plan de Envíos Óptimo ---")
for i in I:
    for j in J:
        print(f"Enviar {x[i, j].varValue} millones de galones de la Refineria {i} al AD {j}")

print(f"\nCosto Total Mínimo: ${pulp.value(model.objective) /100} dólares")

Estado del modelo: Optimal

--- Plan de Envíos Óptimo ---
Enviar 4.0 millones de galones de la Refineria 1 al AD 1
Enviar 2.0 millones de galones de la Refineria 1 al AD 2
Enviar 0.0 millones de galones de la Refineria 1 al AD 3
Enviar 0.0 millones de galones de la Refineria 2 al AD 1
Enviar 5.0 millones de galones de la Refineria 2 al AD 2
Enviar 0.0 millones de galones de la Refineria 2 al AD 3
Enviar 0.0 millones de galones de la Refineria 3 al AD 1
Enviar 1.0 millones de galones de la Refineria 3 al AD 2
Enviar 7.0 millones de galones de la Refineria 3 al AD 3

Costo Total Mínimo: $24.3 dólares


## Pregunta 3

Patito fabrica mochilas para excursionistas exigentes. La demanda de su producto se presenta desde marzo hasta junio de cada año. Patito estima que la demanda durante los cuatro meses es 100, 200, 180 y 300 unidades, respectivamente. La empresa emplea mano de obra de tiempo parcial para fabricar las mochilas y, en consecuencia, su capacidad de producción varía cada mes. Se estima que Patito puede producir 50, 180, 280 y 270 unidades de marzo a junio, respectivamente. Como no coinciden la capacidad de producción y la demanda en los distintos meses, la demanda de determinado mes se puede satisfacer de uno de tres modos: 

La producción del mes en curso.
La producción sobrante en meses anteriores.
La producción sobrante en meses posteriores.
En el primer caso, el costo de producción es 40.00 por mochila. En el segundo se incurre en un costo adicional de retención de 0.50 por mochila por mes. En el tercer caso se incurre en una penalización adicional de 2.00 por mochila y por mes. Patito  desea determinar el programa óptimo de producción en los cuatro meses.

In [32]:
I = [1, 2, 3, 4] # Meses de producción/demanda

c = [
    [40, 40.5, 41, 42.5],
    [42, 40, 40.5, 41], 
    [44, 42, 40, 40.5],
    [46, 44, 42, 40]
]
d = [100, 200, 180, 300]
o = [50, 180, 280, 270]
model = pulp.LpProblem("Mochilas", pulp.LpMinimize)

# Variable de decisión: Cantidad enviada del Silo i al Molino j
x = {(i, j): pulp.LpVariable(f"Mochila_producida_mes_{i}_para_mes_{j}", lowBound=0, cat='Continuous') for i in I for j in I}

# 2. Función Objetivo: Nota el i-1 y j-1 para respetar el índice 0 de Python
model += pulp.lpSum(x[i, j] * c[i-1][j-1] for i in I for j in I), "Costo_Total"

In [34]:
for i in I:
    model += pulp.lpSum(x[i, j] for j in J) <= o[i-1], f"Oferta_{i}"

# 4. Restricción de Demanda: Lo que llega al Molino 'j' debe ser igual a su demanda
for j in J:
    model += pulp.lpSum(x[i, j] for i in I) == d[j-1], f"Demanda_{j}"

# Resolver el problema
model.solve()

# Imprimir los resultados
print(f"Estado del modelo: {pulp.LpStatus[model.status]}\n")
print("--- Plan de Envíos Óptimo ---")
for i in I:
    for j in J:
        print(f"Producir {x[i, j].varValue} de mochilas el mes {i} para el mes {j}")

print(f"\nCosto Total Mínimo: ${pulp.value(model.objective)}")

Estado del modelo: Optimal

--- Plan de Envíos Óptimo ---
Producir 50.0 de mochilas el mes 1 para el mes 1
Producir 0.0 de mochilas el mes 1 para el mes 2
Producir 0.0 de mochilas el mes 1 para el mes 3
Producir 0.0 de mochilas el mes 1 para el mes 4
Producir 50.0 de mochilas el mes 2 para el mes 1
Producir 130.0 de mochilas el mes 2 para el mes 2
Producir 0.0 de mochilas el mes 2 para el mes 3
Producir 0.0 de mochilas el mes 2 para el mes 4
Producir 0.0 de mochilas el mes 3 para el mes 1
Producir 70.0 de mochilas el mes 3 para el mes 2
Producir 180.0 de mochilas el mes 3 para el mes 3
Producir 30.0 de mochilas el mes 3 para el mes 4
Producir 0.0 de mochilas el mes 4 para el mes 1
Producir 0.0 de mochilas el mes 4 para el mes 2
Producir 0.0 de mochilas el mes 4 para el mes 3
Producir 270.0 de mochilas el mes 4 para el mes 4

Costo Total Mínimo: $31455.0


## Problema 4
La Patito Company, que fabrica un solo producto, tiene tres plantas y cuatro clientes. Las plantas respectivas podrán producir 60, 80 y 40 unidades, durante el siguiente periodo. La empresa se ha comprometido a vender 40 unidades al cliente 1, 60 unidades al cliente 2 y por lo menos 20 unidades al cliente 3. Tanto el cliente 3 como el 4 desean comprar tantas unidades como sea posible de las restantes. La ganancia neta asociada con el envío de una unidad de la planta i al cliente j está dada en la tabla:


| Planta | 1 | 2 | 3 | 4 |
| :---: | :---: | :---: |:---: | :---:| 
| 1 |	800 |	700 |500 |	200|
|2	|500	|200	|100	|300|
|3	|600	|400	|300	|500|

La administración desea saber cuántas unidades debe vender a los clientes 3 y 4, y cuántas unidades conviene enviar de cada planta a cada cliente, para maximizar la ganancia.
a) Formule este problema como un problema de transporte donde la función objetivo sea maximizar mediante la construcción de la tabla de parámetros apropiada que proporcione la unidades de ganancia.

In [52]:
I = [1, 2, 3, 4] # Clientes
J = [1, 2, 3, 4] # Plantas

c = [
    [800, 700, 500, 200],
    [500, 200, 100, 300],
    [600, 400, 300, 500],
    [-9999, -9999,  0,   0]  # Planta 4 (Ficticia)
]

o = [60, 80, 40, 60]  
d = [40, 60, 80, 60] 
model = pulp.LpProblem("Patito", pulp.LpMaximize)

# Variable de decisión: Cantidad enviada del Silo i al Molino j
x = {(i, j): pulp.LpVariable(f"Producto_planta_{j}_Cliente_{i}", lowBound=0, cat='Integer') for i in I for j in J}

# 2. Función Objetivo: Nota el i-1 y j-1 para respetar el índice 0 de Python
model += pulp.lpSum(x[i, j] * c[j-1][i-1] for i in I for j in J), "Costo_Total"

In [53]:
# Restricciones de Oferta
for j in J:
    model += pulp.lpSum(x[i, j] for i in I) == o[j-1], f"Oferta_Planta_{j}"

# Restricciones de Demanda
for i in I:
    model += pulp.lpSum(x[i, j] for j in J) == d[i-1], f"Demanda_Cliente_{i}"
    
# Resolver el problema
model.solve()

# Imprimir los resultados
print(f"Estado del modelo: {pulp.LpStatus[model.status]}\n")
print("--- Plan de Envíos Óptimo ---")
for i in I:
    for j in J:
        print(f"Producir {x[i, j].varValue} cantidad de la Planta {j} para el cliente {i}")

print(f"\nCosto Total Mínimo: ${pulp.value(model.objective)}")

Estado del modelo: Optimal

--- Plan de Envíos Óptimo ---
Producir 0.0 cantidad de la Planta 1 para el cliente 1
Producir 40.0 cantidad de la Planta 2 para el cliente 1
Producir 0.0 cantidad de la Planta 3 para el cliente 1
Producir 0.0 cantidad de la Planta 4 para el cliente 1
Producir 60.0 cantidad de la Planta 1 para el cliente 2
Producir 0.0 cantidad de la Planta 2 para el cliente 2
Producir 0.0 cantidad de la Planta 3 para el cliente 2
Producir 0.0 cantidad de la Planta 4 para el cliente 2
Producir 0.0 cantidad de la Planta 1 para el cliente 3
Producir 20.0 cantidad de la Planta 2 para el cliente 3
Producir 0.0 cantidad de la Planta 3 para el cliente 3
Producir 60.0 cantidad de la Planta 4 para el cliente 3
Producir 0.0 cantidad de la Planta 1 para el cliente 4
Producir 20.0 cantidad de la Planta 2 para el cliente 4
Producir 40.0 cantidad de la Planta 3 para el cliente 4
Producir 0.0 cantidad de la Planta 4 para el cliente 4

Costo Total Mínimo: $90000.0
